# Aula 08 - Aprendizado Não Supervisionado parte II

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**10/09/2026 - Sprint 3 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula08.ipynb)

## O que este notebook é

A Aula 06 agrupou os cinco produtos do case com K fixo em 4 e leu a silhueta de dois
agrupamentos, sem escolher K por método nenhum. A Aula 07 fechou com uma regressão linear sobre
11 features mensais, MAPE de teste de 3,32% contra 3,71% da baseline de coeficiente fixo da LDC.
Este notebook mede as duas técnicas que faltavam: escolher K por Elbow Plot e Silhouette
Analysis, e reduzir a dimensionalidade daquelas 11 features por PCA.

A aula tem uma tese única, medida nos dois blocos. O critério interno da técnica não
supervisionada (inércia, silhueta, variância explicada) aponta escolhas diferentes das que o case
precisa, e as células abaixo mostram onde a divergência aparece e quanto ela custa em MAPE.

Escalador e PCA são ajustados **só nos 315 meses de treino**, nunca na base inteira. É a
disciplina que a Aula 05 estabeleceu e que a Aula 09 vai cobrar como vazamento temporal.

## Ao final deste notebook você terá

1. remontado a base de participação de cada trimestre no total do próprio ano, a mesma da
   Aula 06, com 116 linhas;
2. medido inércia, silhueta e concordância com o calendário de K=2 a K=8 nessa base, e visto o
   Elbow apontar K=3, a silhueta apontar K=2 e só K=4 recuperar o trimestre;
3. repetido as três medidas na base de níveis, onde a silhueta é mais alta em todo K e a
   concordância com o calendário não sai do acaso;
4. ajustado escalador e PCA nos 315 meses de treino da base mensal da Aula 07 e lido a variância
   explicada dos 11 componentes;
5. contado os pares de features com correlação absoluta acima de 0,9 e o número de condição da
   matriz padronizada;
6. lido os loadings de PC1, PC2 e PC3 nas 11 colunas originais;
7. projetado os 315 meses de treino no plano PC2 por PC3 e recuperado o mês em 91,7% deles, sem
   que nenhuma coluna da matriz seja o número do mês;
8. medido o MAPE da regressão sobre os k primeiros componentes, de k=1 a k=11, contra a baseline
   da LDC;
9. testado, no desafio, o efeito de ajustar o PCA fora do treino e o que o corte de componentes
   faz com um modelo que decide por distância.

## 1. A base de participação de cada trimestre no ano

A célula abaixo lê as cinco séries trimestrais de `dados/` (abate de bovinos, suínos e frangos,
produção de ovos e de leite) e refaz a base de participação da Aula 06. A definição é a mesma
registrada no projeto da aula, reimplementada aqui porque o notebook precisa rodar sozinho no
Colab, sem depender de nenhum outro arquivo do repositório:

- junção interna das cinco séries por `periodo`, em ordem cronológica, o que dá 117 trimestres,
  de 1997-T1 a 2026-T1;
- só anos com os quatro trimestres medidos entram na conta de participação, o que tira 2026-T1 e
  deixa 116 linhas: incluir um ano incompleto faria o único trimestre dele valer 100% do "ano";
- cada valor vira a fração que representa no total do próprio ano, de modo que os quatro
  trimestres de um mesmo ano somam 1 em qualquer nível de produção;
- `StandardScaler` antes do `KMeans`, sempre com `n_init=50` e `random_state=42`.

A célula traz um `try`/`except` para o caso de a rede da sala cair no meio da leitura: se a
internet falhar e o arquivo não estiver na pasta local, a mensagem de erro orienta a pedir a
pasta `dados` para uma dupla vizinha.

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
SEMENTE = 42
FAIXA_K = range(2, 9)

# mesma resolucao de caminho das aulas anteriores: funciona no repositorio
# clonado (CSVs em ../dados/) e no Colab (baixa da versao publicada)
BASE_LOCAL = os.path.join("..", "dados")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/")


def resolver(pasta_local, url_bruta, nomes):
    """Devolve o caminho de cada CSV, baixando da versao publicada se preciso."""
    caminhos = {}
    for nome in nomes:
        arquivo = nome + ".csv"
        local = os.path.join(pasta_local, arquivo)
        if os.path.exists(local):
            caminhos[nome] = local
        else:
            if not os.path.exists(arquivo):
                try:
                    urllib.request.urlretrieve(url_bruta + arquivo, arquivo)
                except Exception as erro:
                    raise RuntimeError(
                        "Nao foi possivel baixar '%s' pela internet (%s). "
                        "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                        "que tenha o repositorio clonado no computador (ela fica na raiz "
                        "do repositorio) e coloque essa pasta ao lado deste notebook. "
                        "Depois, rode esta celula de novo." % (arquivo, erro)
                    ) from erro
            caminhos[nome] = arquivo
    return caminhos


caminhos = resolver(BASE_LOCAL, BASE_BRUTA, SERIES)

# junta as cinco series por periodo (interseccao, mesma regra da Aula 06)
trimestral = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    trimestral = coluna if trimestral is None else trimestral.merge(
        coluna, on="periodo", how="inner")
trimestral = trimestral.sort_values("periodo").reset_index(drop=True)

X_niveis = trimestral[SERIES].to_numpy(float)
anos = trimestral["periodo"].str[:4].astype(int).to_numpy()
tris = trimestral["periodo"].str[-1].astype(int).to_numpy()

print("base de niveis: %d trimestres, de %s a %s" % (
    len(trimestral), trimestral["periodo"].iloc[0], trimestral["periodo"].iloc[-1]))

# so anos com os quatro trimestres medidos entram na conta de participacao
completos = {a for a in set(anos.tolist()) if (anos == a).sum() == 4}
mascara = np.array([a in completos for a in anos])
X_completos, anos_completos = X_niveis[mascara], anos[mascara]
tris_participacao = tris[mascara]
periodos_participacao = trimestral["periodo"].to_numpy()[mascara]

X_participacao = np.empty_like(X_completos)
for ano in completos:
    linhas = anos_completos == ano
    X_participacao[linhas] = X_completos[linhas] / X_completos[linhas].sum(axis=0)

print("base de participacao no ano: %d trimestres, de %s a %s" % (
    len(X_participacao), periodos_participacao[0], periodos_participacao[-1]))
print("soma dos quatro trimestres de 2025, por serie:",
      np.round(X_participacao[anos_completos == 2025].sum(axis=0), 6))

A base de níveis fica com 117 trimestres e a de participação com 116, exatamente como na
Aula 06. A última linha da célula é uma conferência: a soma das quatro frações de 2025 dá 1 em
cada uma das cinco séries, o que confirma que a normalização foi feita por ano e por coluna, não
sobre a base inteira.

## 2. Inércia, silhueta e concordância de K=2 a K=8

Três medidas entram na tabela da próxima célula, e vale separar o que cada uma responde antes de
ler os números:

- **inércia** é a soma dos quadrados das distâncias de cada ponto ao centroide do próprio
  cluster. Ela cai sempre que K aumenta, porque mais centroides sempre aproximam os pontos de
  algum deles. O Elbow Plot não procura o menor valor: procura o K depois do qual a queda deixa
  de compensar, o "joelho" da curva;
- **silhueta** compara, para cada ponto, a distância média aos pontos do próprio cluster com a
  distância média aos pontos do cluster vizinho mais próximo. Fica entre -1 e 1, e quanto mais
  alta, mais separados os clusters estão entre si;
- **concordância com o trimestre do calendário** é a fração das linhas cobertas pelo trimestre
  majoritário de cada cluster. Ela usa um rótulo que o K-means nunca viu (o trimestre em que cada
  linha aconteceu), e mede o que interessa ao Modelo 2 do TAPI: se o agrupamento reencontrou o
  calendário, ele descreve o padrão sazonal da demanda de ração.

As duas primeiras são critérios internos, calculados só com as colunas que o algoritmo recebeu.
A terceira é o critério do case.

In [ ]:
def concordancia(rotulos, verdade):
    """Fracao das linhas cobertas pelo rotulo majoritario de cada cluster."""
    acertos = 0
    for c in set(rotulos.tolist()):
        do_cluster = verdade[rotulos == c]
        acertos += max((do_cluster == v).sum() for v in set(verdade.tolist()))
    return acertos / len(verdade)


def agrupar(matriz, k):
    """Padroniza, roda o KMeans e devolve inercia, silhueta e rotulos."""
    padronizada = StandardScaler().fit_transform(matriz)
    modelo = KMeans(n_clusters=k, n_init=50, random_state=SEMENTE).fit(padronizada)
    return modelo.inertia_, silhouette_score(padronizada, modelo.labels_), modelo.labels_


print("base de participacao no ano (%d linhas)" % len(X_participacao))
print("%3s %10s %10s %10s %14s" % ("K", "inercia", "queda", "silhueta", "concordancia"))

medidas = {}
anterior = None
for k in FAIXA_K:
    inercia, silhueta, rotulos = agrupar(X_participacao, k)
    conc = concordancia(rotulos, tris_participacao)
    medidas[k] = {"inercia": inercia, "silhueta": silhueta, "concordancia": conc}
    queda = "" if anterior is None else "%.1f%%" % ((anterior - inercia) / anterior * 100)
    print("%3d %10.1f %10s %10.4f %13.1f%%" % (k, inercia, queda, silhueta, conc * 100))
    anterior = inercia

k_silhueta = max(medidas, key=lambda k: medidas[k]["silhueta"])
k_concordancia = max(medidas, key=lambda k: medidas[k]["concordancia"])

print()
print("K de maior silhueta:      %d (%.4f)" % (k_silhueta, medidas[k_silhueta]["silhueta"]))
print("K de maior concordancia:  %d (%.1f%%)" % (
    k_concordancia, medidas[k_concordancia]["concordancia"] * 100))

A tabela fecha o bloco com três respostas diferentes para a mesma pergunta.

A inércia cai 25,1% de K=2 para K=3 e só 14,3% de K=3 para K=4, e depois disso a queda entra num
regime de 12,2%, 10,6%, 7,6% e 6,5%: o joelho da curva está em K=3, e é isso que o Elbow Plot
aponta. A silhueta é máxima em K=2, com 0,3785, e cai para 0,3389 em K=3 e 0,2853 em K=4. A
concordância com o trimestre do calendário vai de 50,0% em K=2 para 75,0% em K=3 e 98,3% em K=4.

O K que serve ao case é 4, o único em que quase toda linha cai no cluster do seu próprio
trimestre. Nenhum dos dois critérios internos aponta esse valor, e o valor que o case precisa tem
a pior silhueta dos três. A silhueta mede corretamente o que ela mede, separação geométrica entre
clusters, e na participação no ano essa separação é sutil por natureza, porque a produção nunca
desaparece num trimestre e dobra no seguinte.

## 3. As mesmas três medidas na base de níveis

A Aula 06 rodou o K-means duas vezes, uma nos níveis de produção e outra na participação no ano,
e registrou o resultado assim: silhueta de 0,4795 no agrupamento de níveis com K=4, com
concordância de 26,5%, contra silhueta de 0,2853 no agrupamento de participação com o mesmo K=4,
com concordância de 98,3%. A célula abaixo estende essa comparação para toda a faixa de K.

In [ ]:
print("base de niveis (%d linhas)" % len(X_niveis))
print("%3s %10s %10s %14s" % ("K", "inercia", "silhueta", "concordancia"))

for k in FAIXA_K:
    inercia, silhueta, rotulos = agrupar(X_niveis, k)
    print("%3d %10.1f %10.4f %13.1f%%" % (
        k, inercia, silhueta, concordancia(rotulos, tris) * 100))

print()
print("acaso, com quatro trimestres possiveis: 25.0%")
print("comparacao em K=4, os dois numeros que a Aula 06 publicou:")
_, silhueta_niveis, rotulos_niveis = agrupar(X_niveis, 4)
_, silhueta_part, rotulos_part = agrupar(X_participacao, 4)
print("  niveis:       silhueta %.4f, concordancia %.1f%%" % (
    silhueta_niveis, concordancia(rotulos_niveis, tris) * 100))
print("  participacao: silhueta %.4f, concordancia %.1f%%" % (
    silhueta_part, concordancia(rotulos_part, tris_participacao) * 100))

Na base de níveis a silhueta é mais alta em todo K, de 0,5639 em K=2 a 0,4247 em K=8, e a
concordância com o calendário fica entre 25,6% e 29,1%, ou seja, no acaso de quatro trimestres
possíveis. Quem escolhesse a base e o K pela silhueta escolheria o agrupamento que não recupera o
calendário em nenhum valor de K testado.

O motivo é o mesmo que a Aula 06 mediu: as cinco séries cresceram ao longo de quase três décadas,
e o nível de produção é dominado por quanto tempo passou. Agrupar níveis produz épocas, e épocas
são grupos bem separados no espaço padronizado. A silhueta premia essa separação, e ela é real: o
que ela não sabe é que o case pergunta outra coisa.

## 4. PCA sobre as 11 features da base mensal

A célula abaixo é **autocontida**: ela lê as cinco séries de `dados/mensal/`, remonta a base
analítica da Aula 07 e ajusta escalador e PCA sozinha, sem depender de nenhuma célula anterior
deste notebook. É a célula da prática do bloco, e nenhuma dupla precisa ter rodado as seções 1 a
3 para chegar aqui.

A base é a mesma da Aula 07, com a mesma definição:

- junção interna das cinco séries mensais por `periodo`, em ordem cronológica;
- `mes` é o inteiro dos dois últimos caracteres de `periodo`; `dias` é o número de dias do mês
  civil;
- `sen` e `cos` são o par de sazonalidade, `sin(2*pi*mes/12)` e `cos(2*pi*mes/12)`;
- `lag1`, `lag2`, `lag3` e `lag12` são `abate_frangos` defasado em 1, 2, 3 e 12 meses;
- `<serie>_lag1` são as outras quatro séries defasadas em 1 mês;
- linhas com qualquer valor ausente são descartadas, o que tira os 12 primeiros meses da série
  por causa de `lag12` e deixa 339 linhas, de 1998-01 a 2026-03.

As `FEATURES` são as 11 do modelo do fecho da Aula 07, aquele que bateu a baseline da LDC com
MAPE de 3,32%. O corte é por data: 315 meses de treino (1998-01 a 2024-03) e 24 meses de teste
(2024-04 a 2026-03). `StandardScaler` e `PCA` veem **apenas as 315 primeiras linhas**.

In [ ]:
import calendar
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"
FEATURES = (["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
            + [serie + "_lag1" for serie in SERIES if serie != ALVO])
N_TESTE = 24
SEMENTE = 42

MENSAL_LOCAL = os.path.join("..", "dados", "mensal")
MENSAL_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
                "main/dados/mensal/")

# esta celula e autocontida: repete a resolucao de caminho para nao depender
# da secao 1 (ADR-011, mitigacao do risco de a pratica cair junto com uma
# celula anterior)
caminhos_mensais = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(MENSAL_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos_mensais[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(MENSAL_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos_mensais[nome] = arquivo

base = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos_mensais[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)

base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)
base = base.dropna().reset_index(drop=True)

CORTE = len(base) - N_TESTE
print("base mensal: %d linhas, de %s a %s" % (
    len(base), base["periodo"].iloc[0], base["periodo"].iloc[-1]))
print("treino: %d meses (%s a %s)" % (
    CORTE, base["periodo"].iloc[0], base["periodo"].iloc[CORTE - 1]))
print("teste: %d meses (%s a %s)" % (
    N_TESTE, base["periodo"].iloc[CORTE], base["periodo"].iloc[-1]))

# escalador e PCA ajustados SO no treino (ADR-011)
X = base[FEATURES].to_numpy(float)
escalador = StandardScaler().fit(X[:CORTE])
Ztr, Zte = escalador.transform(X[:CORTE]), escalador.transform(X[CORTE:])
pca = PCA().fit(Ztr)

variancia = pca.explained_variance_ratio_
acumulada = np.cumsum(variancia)

print()
print("%-6s %12s %12s" % ("comp", "variancia", "acumulada"))
for i in range(len(FEATURES)):
    print("%-6s %11.2f%% %11.2f%%" % ("PC%d" % (i + 1),
                                      variancia[i] * 100, acumulada[i] * 100))

print()
print("componentes para chegar a 95%% da variancia: %d"
      % (int(np.searchsorted(acumulada, 0.95)) + 1))

PC1 concentra 68,21% da variância das 11 colunas padronizadas. PC2 vale 11,85% e PC3 vale
9,24%, o que leva a acumulada a 80,06% em dois componentes e a 96,07% em quatro. Quatro direções,
das onze disponíveis, passam de 95% da variância.

Esse é o número que costuma decidir um corte de dimensionalidade na prática: onze colunas viram
quatro e "só" 3,93% da variância fica de fora. A seção 8 mede o que esse corte custa em MAPE.

## 5. Vinte dos 55 pares de features têm correlação absoluta acima de 0,9

A pergunta disparada da aula é quantas das 11 features são realmente independentes entre si. A
célula abaixo responde com duas medidas sobre a matriz padronizada de treino: a contagem de pares
com correlação absoluta acima de 0,9, e o número de condição, a razão entre o maior e o menor
valor singular da matriz. Número de condição alto significa que existe combinação de colunas
quase redundante, o mesmo diagnóstico que a Aula 05 usou para condicionamento.

In [ ]:
import itertools

correlacao = np.corrcoef(Ztr, rowvar=False)
pares = [(FEATURES[i], FEATURES[j], correlacao[i, j])
         for i, j in itertools.combinations(range(len(FEATURES)), 2)]
altos = [p for p in pares if abs(p[2]) > 0.9]

print("pares de features: %d" % len(pares))
print("pares com correlacao absoluta acima de 0,9: %d" % len(altos))
print()
print("os cinco pares de maior correlacao absoluta:")
for a, b, r in sorted(pares, key=lambda p: -abs(p[2]))[:5]:
    print("  %-20s %-20s %7.4f" % (a, b, r))

print()
print("numero de condicao da matriz padronizada de treino: %.2f"
      % np.linalg.cond(Ztr))

Vinte dos 55 pares passam de 0,9 de correlação absoluta, e o par mais correlacionado é
`lag1` com `lag3`, em 0,9810. O número de condição da matriz padronizada de treino é 28,50.

As 11 colunas não são 11 direções independentes de informação: as oito colunas em quilos (as
quatro defasagens de `abate_frangos` e as quatro defasagens das outras séries) sobem e descem
quase juntas, porque todas carregam o mesmo crescimento de longo prazo. É essa redundância que o
PCA encontra e comprime em PC1.

## 6. Os loadings de PC1, PC2 e PC3

Cada componente é uma combinação linear das 11 colunas padronizadas, e os pesos dessa combinação
são os loadings. Ler os loadings é a única forma de dizer o que um componente significa em termos
das colunas originais.

In [ ]:
EM_QUILOS = (["lag1", "lag2", "lag3", "lag12"]
             + [serie + "_lag1" for serie in SERIES if serie != ALVO])
DO_CALENDARIO = ["sen", "cos", "dias"]

print("%-22s %10s %10s %10s" % ("feature", "PC1", "PC2", "PC3"))
for i, nome in enumerate(FEATURES):
    print("%-22s %10.3f %10.3f %10.3f" % (
        nome, pca.components_[0][i], pca.components_[1][i], pca.components_[2][i]))

peso1 = dict(zip(FEATURES, np.abs(pca.components_[0])))
print()
print("PC1, peso absoluto nas oito colunas em quilos: de %.3f a %.3f" % (
    min(peso1[f] for f in EM_QUILOS), max(peso1[f] for f in EM_QUILOS)))
print("PC1, maior peso absoluto em sen, cos e dias:   %.4f" % max(
    peso1[f] for f in DO_CALENDARIO))

PC1 pesa entre 0,336 e 0,362 em cada uma das oito colunas em quilos, e no máximo 0,0196 em
`sen`, `cos` e `dias`. É o nível comum das cinco séries, a tendência de 28 anos que todas
compartilham, e nada mais.

PC2 e PC3 carregam o calendário. PC2 pesa 0,692 em `dias`, -0,619 em `sen` e -0,341 em `cos`.
PC3 pesa 0,885 em `cos` e -0,432 em `sen`. Nenhum dos dois pesa mais do que 0,13 em qualquer
coluna em quilos.

A leitura tem consequência prática para o case: o primeiro componente, o de maior variância, não
tem nenhuma informação de sazonalidade. Quem retiver só PC1 fica com a tendência e joga fora todo
o calendário, que é justamente o que o Modelo 2 do TAPI precisa.

## 7. Doze grupos no plano PC2 por PC3 recuperam o mês em 91,7% dos 315 meses de treino

Se PC2 e PC3 carregam o calendário, os meses precisam aparecer separados no plano formado por
eles. A célula abaixo projeta os 315 meses de treino nesse plano, roda um K-means com 12 grupos e
mede a concordância com o mês verdadeiro, além da silhueta dos rótulos verdadeiros de mês. Vale
lembrar que nenhuma coluna da matriz é o número do mês: entram `sen`, `cos` e o número de dias.

In [ ]:
import matplotlib.pyplot as plt

# cores lidas de assets/css/inteli-brand.css
TINTA, DESTAQUE, NUVEM = "#2e2640", "#ff4545", "#caced6"


def concordancia_mes(rotulos, verdade):
    """Fracao das linhas cobertas pelo rotulo majoritario de cada cluster."""
    acertos = 0
    for c in set(rotulos.tolist()):
        do_cluster = verdade[rotulos == c]
        acertos += max((do_cluster == v).sum() for v in set(verdade.tolist()))
    return acertos / len(verdade)


plano = pca.transform(Ztr)[:, 1:3]
meses_treino = base["mes"].to_numpy()[:CORTE]

rotulos_plano = KMeans(n_clusters=12, n_init=50, random_state=SEMENTE).fit_predict(plano)
print("12 grupos no plano PC2xPC3, concordancia com o mes: %.1f%%"
      % (concordancia_mes(rotulos_plano, meses_treino) * 100))
print("silhueta dos rotulos verdadeiros de mes nesse plano: %.4f"
      % silhouette_score(plano, meses_treino))

centroides = np.array([plano[meses_treino == m].mean(axis=0) for m in range(1, 13)])
ciclo = np.vstack([centroides, centroides[:1]])

fig, eixo = plt.subplots(figsize=(8, 6))
eixo.scatter(plano[:, 0], plano[:, 1], s=16, color=NUVEM)
eixo.plot(ciclo[:, 0], ciclo[:, 1], color=TINTA, linewidth=1.4, zorder=2)
for mes, (x, y) in enumerate(centroides, start=1):
    eixo.annotate("%d" % mes, (x, y), color=DESTAQUE, fontsize=13,
                  fontweight="bold", ha="center", va="center", zorder=3)
eixo.set_xlabel("PC2 (11,85% da variância)")
eixo.set_ylabel("PC3 (9,24% da variância)")
eixo.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

print()
print("posicao em PC2 do centroide de cada mes, do menor para o maior:")
ordem = np.argsort(centroides[:, 0])
for i in ordem:
    print("  mes %2d   PC2 %7.3f   dias no mes: %s" % (
        i + 1, centroides[i, 0],
        sorted(set(base["dias"].to_numpy()[:CORTE][meses_treino == i + 1].tolist()))))

A concordância dos 12 grupos com o mês verdadeiro é de 91,7%, e a silhueta dos rótulos
verdadeiros de mês nesse plano é 0,8037. Os doze meses aparecem em ciclo na figura, com o
centroide de cada mês ligado ao do mês seguinte, e fevereiro fica isolado à esquerda, no extremo
negativo de PC2, porque é o mês com 28 ou 29 dias contra os 30 ou 31 dos outros.

O PCA recebeu `sen`, `cos` e `dias`, nunca o número do mês. A estrutura do calendário estava
nessas três colunas, e a rotação a colocou em duas direções, PC2 e PC3, que juntas valem 21,09%
da variância.

## 8. O corte de componentes custa 1,6 ponto percentual de MAPE

Até aqui o PCA descreveu a base. A célula abaixo mede o que ele faz com a previsão. Para cada k
de 1 a 11, a regressão linear é treinada sobre os k primeiros componentes do treino, com o mesmo
alvo em razão do modelo do fecho da Aula 07 (`abate_frangos` dividido por `lag12`), e a previsão
é multiplicada de volta pelo `lag12` do mês de teste. A referência é a baseline C da Aula 07, o
coeficiente fixo que a LDC usa hoje.

In [ ]:
def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


razao_treino = (base[ALVO] / base["lag12"]).to_numpy()[:CORTE]
y_teste = base[ALVO].to_numpy()[CORTE:]
lag12_teste = base["lag12"].to_numpy()[CORTE:]

# baseline C da Aula 07: lag12 vezes o fator medio medido so no treino
fator = float(np.mean(base[ALVO].to_numpy()[:CORTE] / base["lag12"].to_numpy()[:CORTE]))
baseline_ldc = lag12_teste * fator
mape_baseline = mape(y_teste, baseline_ldc)


def mape_com_k(k):
    """MAPE de teste da regressao sobre os k primeiros componentes."""
    reduzido = PCA(n_components=k).fit(Ztr)
    modelo = LinearRegression().fit(reduzido.transform(Ztr), razao_treino)
    previsto = modelo.predict(reduzido.transform(Zte)) * lag12_teste
    return mape(y_teste, previsto)


# o modelo do fecho da Aula 07: as 11 features padronizadas, sem PCA
sem_pca = LinearRegression().fit(Ztr, razao_treino)
mape_sem_pca = mape(y_teste, sem_pca.predict(Zte) * lag12_teste)

print("baseline C da LDC (coeficiente fixo):     MAPE %.2f%%" % mape_baseline)
print("modelo do fecho da Aula 07, 11 features:  MAPE %.2f%%" % mape_sem_pca)
print()
print("%3s %14s %10s %22s" % ("k", "var. acumulada", "MAPE", "contra a baseline"))
for k in range(1, len(FEATURES) + 1):
    m = mape_com_k(k)
    veredito = "perde" if m > mape_baseline else "ganha"
    print("%3d %13.2f%% %9.2f%% %22s" % (k, acumulada[k - 1] * 100, m, veredito))

print()
print("diferenca entre k=11 e as 11 features sem PCA: %.2e ponto percentual"
      % abs(mape_com_k(len(FEATURES)) - mape_sem_pca))

O modelo das 11 features tem MAPE de 3,32% e ganha da baseline da LDC, que está em 3,71%.
Reduzir a dimensionalidade derruba essa vantagem em todo k de 1 a 9: o MAPE é 4,92% com um e com
dois componentes, 4,94% com quatro (os mesmos quatro que retêm 96,07% da variância) e 6,20% com
nove. Só k=10 devolve os 3,32%.

Com k=11 o MAPE é idêntico ao das 11 features sem PCA até a nona casa decimal, e a última linha
da célula mostra a diferença na casa de 1e-15. PCA sem descarte é uma rotação, e a Aula 05 já
havia medido que regressão linear sem regularização é invariante a transformação afim das
entradas. O corte de componentes é o que custa MAPE, e a rotação sai de graça.

**Escopo desta conclusão.** Ela vale para regressão linear sem regularização, estas 11 features,
339 linhas mensais e alvo em razão. Não é uma afirmação geral sobre PCA. Existem modelos em que a
redução compensa, e o próprio acervo tem um: o KNN da Aula 07, cuja distância euclidiana soma a
diferença de todas as colunas, é sensível ao número de colunas. A Aula 09 volta a esse ponto no
bloco de maldição de dimensionalidade.

## 9. PC9 e PC10 recebem os dois maiores coeficientes da regressão

A seção anterior mostrou que o corte custa MAPE, e a próxima célula mostra por quê: imprime o
coeficiente que a regressão sobre os 11 componentes dá a cada um deles, ao lado da variância que
cada componente explica.

In [ ]:
coeficientes = LinearRegression().fit(
    pca.transform(Ztr), razao_treino).coef_

print("%-6s %12s %14s %18s" % ("comp", "variancia", "coeficiente", "vezes o do PC1"))
for i in range(len(FEATURES)):
    print("%-6s %11.2f%% %14.6f %17.1f" % (
        "PC%d" % (i + 1), variancia[i] * 100, coeficientes[i],
        abs(coeficientes[i]) / abs(coeficientes[0])))

maiores = np.argsort(-np.abs(coeficientes))[:2]
print()
print("os dois maiores coeficientes em modulo estao em: %s" % ", ".join(
    "PC%d" % (i + 1) for i in maiores))

PC1 vale 68,21% da variância e recebe coeficiente -0,009222. PC9 vale 0,21% e recebe
-0,203820, 22 vezes o coeficiente de PC1. PC10 vale 0,17% e recebe 0,250588, 27 vezes o de PC1.
Os dois componentes de menor variância entre os que sobram são exatamente os que a regressão mais
usa.

Isso explica a tabela da seção 8 linha por linha: cortar em k=4 elimina PC9 e PC10, que são os
componentes de que o modelo mais depende. A causa é estrutural: o PCA nunca vê o alvo, então ele
ordena as direções por dispersão das entradas, sem saber quais delas guardam relação com a saída.
Essa relação é a medida que decide o MAPE.

## 10. Sem padronizar, PC1 sobe para 96,72% e passa a descrever a unidade das colunas

A seção 4 padronizou antes de rodar o PCA. A célula abaixo repete o PCA sobre a mesma matriz de
treino, agora em unidades originais, para medir o que a padronização estava evitando.

In [ ]:
pca_cru = PCA().fit(X[:CORTE])
peso_cru = dict(zip(FEATURES, np.abs(pca_cru.components_[0])))
desvios = dict(zip(FEATURES, X[:CORTE].std(axis=0)))

print("PC1 sem padronizar: %.2f%% da variancia" % (pca_cru.explained_variance_ratio_[0] * 100))
print("PC1 padronizado:    %.2f%% da variancia" % (variancia[0] * 100))
print()
print("os quatro maiores pesos de PC1 sem padronizar:")
for nome in sorted(peso_cru, key=peso_cru.get, reverse=True)[:4]:
    print("  %-22s %.3f" % (nome, peso_cru[nome]))
print()
print("peso de PC1 sem padronizar nas colunas de calendario:")
for nome in DO_CALENDARIO:
    print("  %-22s %.2e" % (nome, peso_cru[nome]))
print()
print("desvio-padrao no treino, em unidade original:")
for nome in ("lag1", "sen"):
    print("  %-22s %.3f" % (nome, desvios[nome]))

Sem padronizar, PC1 explica 96,72% da variância, e seus maiores pesos são `lag12` com
0,481, `lag3` com 0,475 e `lag1` com 0,475, todas colunas em quilos. `sen`, `cos` e `dias` ficam
com peso abaixo de 1e-6, ou seja, não participam.

A causa está na unidade de medida: o desvio-padrão de `lag1` no treino é 244.980.297, e o de
`sen` é 0,708. A variância total da matriz não padronizada é quase toda das colunas em
quilogramas, e o PCA, que maximiza variância, encontra a unidade de medida antes de encontrar
qualquer estrutura dos dados. PC1 chega a 96,72% descrevendo a escala das colunas, e um número
alto de variância explicada não garante nada sobre a estrutura por trás dele. É o mesmo achado de
condicionamento da Aula 05, agora em outro método.

## 11. Desafio

Duas perguntas para responder rodando código, sem consultar o material de apoio antes de tentar.
As respostas estão registradas lá, para conferência depois.

1. A seção 4 ajusta `StandardScaler` e `PCA` apenas nas 315 primeiras linhas. Refaça o ajuste dos
   dois sobre as 339 linhas da base inteira, mantendo o mesmo corte na hora de treinar a
   regressão e de prever o teste, e compare a variância de PC1 e o MAPE de k=4 com os valores
   desta aula (68,21% e 4,94%). O efeito é grande ou pequeno, e em que direção ele aponta?
2. A seção 8 mede o custo do corte com `LinearRegression`. Repita a medição trocando o modelo por
   `KNeighborsRegressor(n_neighbors=5)` sobre os mesmos componentes, e compare com o MAPE do
   mesmo KNN treinado sobre as 11 features padronizadas, sem PCA. O corte de componentes custa
   MAPE nesse modelo também?

In [ ]:
# 1. ajustar escalador e PCA na base inteira, e nao so no treino
# dica: refaca StandardScaler().fit(X) e PCA().fit(...) sobre as 339 linhas,
# e depois treine a regressao ainda com as 315 primeiras


# 2. o corte de componentes com um modelo que decide por distancia
# dica: from sklearn.neighbors import KNeighborsRegressor, e reaproveite
# mape_com_k trocando o modelo de dentro


resposta_1 = "..."
resposta_2 = "..."